# Primer Parcial
Predecir si un alumno puede aprobar en uno cualquiera de los finales que rinde sabiendo su firma.

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Cargando dataset...")
csv_path = r'c:\Users\marti\Documents\proyectos_ML\Primer_Parcial\reglamento_nuevo_unificado.csv'
df_raw = pd.read_csv(csv_path)
print("Dataset cargado exitosamente")

career_code_mapping = {
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    'ELE-PLS13': 'Ing. Electromecánica', 'ELE-PLS23': 'Ing. Electromecánica',
    'INT9ELECTR': 'Ing. Electromecánica', 'INT9SDIGYT': 'Ing. Electromecánica',
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    'ECA-PLS13': 'Ing. Electrónica', 'ECA-PLS23': 'Ing. Electrónica', 'ECA9-OPT': 'Ing. Electrónica'
}

df_clean = df_raw.copy()
df_clean['Carrera'] = df_clean['Firma'].astype(str).str.strip().map(career_code_mapping)

s_nota = df_clean['Nota.Final'].fillna('')
df_clean['Rendio_1F'] = s_nota.str.contains('1F-').astype(int)
df_clean['Nota_1F'] = s_nota.str.extract(r'1F-(\d)')[0].astype(float)
df_clean['Aprobo_1F'] = (df_clean['Nota_1F'] >= 2).astype(int)

df_clean['Rendio_2F'] = s_nota.str.contains('2F-').astype(int)
df_clean['Nota_2F'] = s_nota.str.extract(r'2F-(\d)')[0].astype(float)
df_clean['Aprobo_2F'] = (df_clean['Nota_2F'] >= 2).astype(int)

# Filtrado de los estudiantes con firma.
df_firma = df_clean[df_clean['Firma'] > 0].copy()

# Parámteros a usar.
features_num = ['Firma', 'Primer.Par', 'Segundo.Par', 'Tercer.Par', 'TPLab.', 'Lab.', 'Proy.', 'Asis', 'Pond.PP', 'Pond.SP']
X = df_firma[features_num].fillna(0)

#Modelo 1: Predecir si aprobó el primer final.
df_1f = df_firma[df_firma['Rendio_1F'] == 1].copy()
X_1f = df_1f[features_num].fillna(0)
y_1f = df_1f['Aprobo_1F']

X_train_1f, X_test_1f, y_train_1f, y_test_1f = train_test_split(X_1f, y_1f, test_size=0.2, random_state=42)

model_1f = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model_1f.fit(X_train_1f, y_train_1f)

y_pred_1f = model_1f.predict(X_test_1f)
y_prob_1f = model_1f.predict_proba(X_test_1f)[:, 1]

print("=== EVALUACION MODELO 1F ===")
print("ROC-AUC:", roc_auc_score(y_test_1f, y_prob_1f))
print(classification_report(y_test_1f, y_pred_1f))

#Modelo 2: Predecir si aprobó el segundo final sin haber rendido el primer.
df_2f_dir = df_firma[(df_firma['Rendio_1F'] == 0) & (df_firma['Rendio_2F'] == 1)].copy()
X_2f_dir = df_2f_dir[features_num].fillna(0)
y_2f_dir = df_2f_dir['Aprobo_2F']

X_train_2f, X_test_2f, y_train_2f, y_test_2f = train_test_split(X_2f_dir, y_2f_dir, test_size=0.2, random_state=42)

model_2f = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model_2f.fit(X_train_2f, y_train_2f)

y_pred_2f = model_2f.predict(X_test_2f)
y_prob_2f = model_2f.predict_proba(X_test_2f)[:, 1]

print("=== EVALUACION MODELO 2F DIRECTO ===")
print("ROC-AUC:", roc_auc_score(y_test_2f, y_prob_2f))
print(classification_report(y_test_2f, y_pred_2f))

#Definición del método recomendar_final.
def recomendar_final(alumno_dict):
    df_alumno = pd.DataFrame([alumno_dict])[features_num].fillna(0)
    p_1f = model_1f.predict_proba(df_alumno)[0, 1]
    p_2f = model_2f.predict_proba(df_alumno)[0, 1]
    
    if p_1f >= 0.50:
        rec = "RENDIR 1º FINAL"
        just = f"Tienes una probabilidad estimada del {p_1f*100:.1f}% de aprobar en 1F. Además conservas 2º Final como respaldo."
    elif p_2f > p_1f + 0.10:
        rec = "ESPERAR AL 2º FINAL"
        just = f"Tu probabilidad en 1F es baja ({p_1f*100:.1f}%), pero se estima en {p_2f*100:.1f}% para 2F. Te conviene disponer de más semanas de preparación."
    else:
        rec = "REFORZAR INTENSIVAMENTE"
        just = f"Probabilidades bajas en ambos finales (1F: {p_1f*100:.1f}%, 2F: {p_2f*100:.1f}%). Requiere repaso profundo antes de presentar examen."
        
    return {
        'Recomendacion': rec,
        'Prob_1F': round(p_1f * 100, 1),
        'Prob_2F_Directo': round(p_2f * 100, 1),
        'Justificacion': just
    }

#Prueba casos simples.
print("\n=== PRUEBA DE CASOS DE ALUMNOS ===")
case_1 = {'Firma': 85, 'Primer.Par': 80, 'Segundo.Par': 85, 'Tercer.Par': 0, 'TPLab.': 100, 'Lab.': 100, 'Proy.': 0, 'Asis': 1, 'Pond.PP': 30, 'Pond.SP': 35}
print("Alumno Firma Alta (85):", recomendar_final(case_1))

case_2 = {'Firma': 42, 'Primer.Par': 35, 'Segundo.Par': 45, 'Tercer.Par': 0, 'TPLab.': 70, 'Lab.': 70, 'Proy.': 0, 'Asis': 1, 'Pond.PP': 15, 'Pond.SP': 20}
print("Alumno Firma Baja (42):", recomendar_final(case_2))

